# Conv-TasNet

In this notebook, you will learn how to construct a full architecture of `Conv-TasNet` model and use a given dataset (Libri2Mix) to finish the training process and evaluation

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchaudio
import os

## Step 1: Loading the dataset to Dataloader

First, let's prepare the dataloader, we should define a dedicated class to read the dataset

The `_load_audio` function does the following things:
- Load the waveform
- Convert the audio to mono
- Resample the audio to the given sample rate
- Fix the length

The `_getitem_` function receives an index to load the mixture audio and its corresponding ground truth (the separated sources)

In [2]:
class SpeechDataset(Dataset):
  def __init__(self, mix_dir, s1_dir, s2_dir, sample_rate=16000, num_samples=32000):
    self.mix_dir = mix_dir
    self.s1_dir = s1_dir
    self.s2_dir = s2_dir
    self.sample_rate = sample_rate
    self.num_samples = num_samples

    # Assuming all files have the same name for mix, s1, and s2
    self.ids = [os.path.splitext(f)[0] for f in os.listdir(mix_dir) if f.endswith('.wav')]

  def __len__(self):
    return len(self.ids)

  def _load_audio(self, path):
    wav, sr = torchaudio.load(path)
    wav = wav.mean(0)  # Convert to mono if necessary
    if sr != self.sample_rate:
      wav = torchaudio.functional.resample(wav, sr, self.sample_rate)
    if wav.shape[0] < self.num_samples:
      wav = F.pad(wav, (0, self.num_samples - wav.shape[0]))
    else:
      wav = wav[:self.num_samples]
    return wav

  def __getitem__(self, idx):
    utt_id = self.ids[idx]
    mix_wav = self._load_audio(os.path.join(self.mix_dir, utt_id + '.wav'))
    s1_wav  = self._load_audio(os.path.join(self.s1_dir, utt_id + '.wav'))
    s2_wav  = self._load_audio(os.path.join(self.s2_dir, utt_id + '.wav'))
    sources = torch.stack([s1_wav, s2_wav], dim=0)  # (2, T)
    return mix_wav, sources

Download the dataset called `Libri2Mix.zip` in Moodle. It includes 2 folders: `train-100`, and `test`

The `train-100` folder includes 3 folders:
- `mix_clean`: All mixture audios
- `s1`: Speaker 1 (separated source 1)
- `s2`: Speaker 2 (separated source 2)


### ***Task***: *Complete the code below to prepare the training dataset and the testing dataset by doing the following steps:*
1. filling in the directory path of `mix_clean`, `s1`, and `s2` for the ***traning*** and the ***testing*** dataset, respectively. 
2. Assign the parameter `sample_rate` with approiate value when creating the training and testing dataset

In [3]:
num_samples = 40000

# Filling in the path of following 3 directories for the training dataset
mix_train_dir = "./Dataset/Libri2Mix/train/mix_clean"
s1_train_dir = "./Dataset/Libri2Mix/train/s1"
s2_train_dir = "./Dataset/Libri2Mix/train/s2"

# Add the parameter "sample_rate" and assign an appropriate value to it
train_dataset = SpeechDataset(
  mix_train_dir,
  s1_train_dir,
  s2_train_dir,
  sample_rate=8000,
  num_samples=num_samples
)

# Filling in the path of following 3 directories for the testing dataset
# mix_test_dir = "xxx"
# s1_test_dir = "xxx"
# s2_test_dir = "xxx"

# # Add the parameter "sample_rate" and assign an appropriate value to it
# test_dataset = SpeechDataset(
#   mix_test_dir,
#   s1_test_dir,
#   s2_test_dir,
#   num_samples=num_samples
# )

Then, we can create the training dataloader. You can adjust the `batch_size` yourself. (No need to create the testing dataloader, as we won't use all testing signals for evaluation)

In [4]:
from torch.utils.data import DataLoader

batch_size = 4

trainloader = DataLoader(
  train_dataset,
  batch_size=batch_size,
  shuffle=True
)

## Step 2: Define the network architecture

The ***Conv-TasNet*** contains 3 parts: `Encoder`, `Separator`, and `Decoder`. The most important part is the `Separator`, and this parts contains several `TCN` (Temporal Convolutional Network). Below are symbols used in the network

<img src="./Images/hyperparameters.png" width=480 />

In [5]:
class Conv1DBlock(nn.Module):
  def __init__(self, in_ch, hidden_ch, kernel_size, dilation):
    super().__init__()
    self.conv1 = nn.Conv1d(in_ch, hidden_ch, 1)
    self.prelu1 = nn.PReLU()
    self.norm1 = nn.GroupNorm(1, hidden_ch)
    self.dwconv = nn.Conv1d(hidden_ch, hidden_ch, kernel_size,
                            padding=(kernel_size-1)//2 * dilation,
                            dilation=dilation, groups=hidden_ch)
    self.prelu2 = nn.PReLU()
    self.norm2 = nn.GroupNorm(1, hidden_ch)
    self.resconv = nn.Conv1d(hidden_ch, in_ch, 1)

  def forward(self, x):
    out = self.conv1(x)
    out = self.prelu1(out)
    out = self.norm1(out)
    out = self.dwconv(out)
    out = self.prelu2(out)
    out = self.norm2(out)
    out = self.resconv(out)
    return out + x

class Separator(nn.Module):
  def __init__(self, N=256, B=256, H=512, P=3, X=6, R=2, num_spks=2):
    super().__init__()
    self.layernorm = nn.GroupNorm(1, N)
    self.bottleneck = nn.Conv1d(N, B, 1)
    self.TCN = nn.ModuleList()
    for r in range(R):
      for x in range(X):
        self.TCN.append(Conv1DBlock(B, H, P, dilation=2**x))
    self.mask_conv = nn.Conv1d(B, num_spks * N, 1)
    self.N = N
    self.num_spks = num_spks

  def forward(self, x):
    x = self.layernorm(x)
    x = self.bottleneck(x)
    for block in self.TCN:
      x = block(x)
    masks = self.mask_conv(x)
    B, _, T = masks.shape
    masks = masks.view(B, self.num_spks, self.N, T)
    masks = F.relu(masks)
    return masks

class ConvTasNet(nn.Module):
  def __init__(self, N=256, L=20, **kwargs):
    super().__init__()
    self.encoder = nn.Conv1d(1, N, L, stride=L // 2, bias=False)
    self.separator = Separator(N=N, **kwargs)
    self.decoder = nn.ConvTranspose1d(N, 1, L, stride=L // 2, bias=False)

  def forward(self, mixture):
    x = mixture
    if mixture.dim() == 1:
      raise Exception("The dimension of the input must be at least 2")

    if mixture.dim() == 2:
      x = mixture.unsqueeze(1)  # (B, 1, T)

    enc = self.encoder(x)     # (B, N, T')
    masks = self.separator(enc)  # (B, C, N, T')
    masked = masks * enc.unsqueeze(1)  # (B, C, N, T')
    B, C, N, T = masked.shape
    masked = masked.view(B*C, N, T)
    decoded = self.decoder(masked)  # (B*C, 1, T_out)
    decoded = decoded.squeeze(1)
    decoded = decoded.view(B, C, -1)  # (B, C, T_out)
    return decoded

## Step 3: Define the loss function

***SI-SNR (Scale-Invariant Signal-to-Noise Ratio)*** is a common loss/metric for speech separation models. It measures how similar the separated speech is to the clean reference speech while ignoring differences in overall amplitude (scale)

In [6]:
# Loss function
def si_snr(est, ref, eps=1e-8):
  ref = ref - ref.mean(dim=1, keepdim=True)
  est = est - est.mean(dim=1, keepdim=True)
  s_target = (torch.sum(est * ref, dim=1, keepdim=True) * ref) / (
      torch.sum(ref ** 2, dim=1, keepdim=True) + eps)
  e_noise = est - s_target
  si_snr_val = 10 * torch.log10(
      (torch.sum(s_target ** 2, dim=1) + eps) /
      (torch.sum(e_noise ** 2, dim=1) + eps))
  return si_snr_val

def pit_loss(ests, refs):
  # ests/refs: (B, 2, T)
  loss1 = si_snr(ests[:, 0], refs[:, 0]) + si_snr(ests[:, 1], refs[:, 1])
  loss2 = si_snr(ests[:, 0], refs[:, 1]) + si_snr(ests[:, 1], refs[:, 0])
  # maximize SI-SNR, so minimize negative SI-SNR
  return -torch.mean(torch.stack([loss1, loss2], dim=0).max(dim=0)[0]) / 2

## Step 4: Create an instance of model

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

convtasnet = ConvTasNet().to(device)
optimizer = torch.optim.Adam(
  convtasnet.parameters(),
  lr=0.001
)

## Step 5: Train the model

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
  convtasnet.train()
  running_loss = 0

  print(f"Epoch: {epoch + 1}/{num_epochs}")
  for i, (mix, sources) in enumerate(trainloader):
    mix = mix.to(device)
    sources = sources.to(device)

    est_sources = convtasnet(mix)   # (B, 2, T)
    min_len = min(est_sources.shape[-1], sources.shape[-1])
    est_sources = est_sources[:, :min_len]
    sources = sources[:, :min_len]
    
    loss = pit_loss(est_sources, sources)
    print(f"\rLoader: {i + 1}/{len(trainloader)}  Loss: {loss.item()}", end="", flush=True)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    running_loss += loss.item()

  avg_loss = running_loss / len(trainloader)
  print(f'\nAverage Loss in Epoch {epoch + 1}: {avg_loss:.4f}')
  print('------------------------------')

### ***Task:*** Save the model as `pth` file by running the following code and submit the saved model to Moodle

Save the file name as `ConvTasNet-xxxxxxxx.pth`, where `xxxxxxxx` is your student ID

In [ ]:
torch.save(convtasnet.state_dict(), 'ConvTasNet-xxxxxxxx.pth')

## Step 6: Evaluate the model

### ***Task***: *Evaluate the model on the three provided test audio files by computing SDR, SIR, and SAR. Visualize the waveforms and spectrograms of the model’s estimated source signals*

> Use the last 3 digits of your student ID as index of the testing dataset. For example, if your student ID is 26123**456**, the last 3 digits is **456**. You should access the testing mixture signal and its sources using `test_data[456]`.

In [ ]:
import IPython.display as ipd

contasnet_eval = ConvTasNet().to(device)
contasnet_eval.load_state_dict(torch.load("Conv-TasNet-Customized.pth", weights_only=True))

last_3_digits = 123  # replace this value yourself

contasnet_eval.eval()
with torch.no_grad():
  # mix, sources = test_dataset[last_3_digits]

  """ Complete the code below yoursefl """
